# Capability 10 — Decision engine and explanations

Executed evidence over in-memory seed facts. Churn is a constraint, not a discount. No sklearn.


In [1]:
import json
from datetime import datetime, timedelta
from decimal import Decimal
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from telco_digital.application.clock import FixedClock
from telco_digital.application.seed import seed_demo_customers
from telco_digital.decisioning import DecisionEngine
from telco_digital.infrastructure.memory import InMemoryUnitOfWork
from telco_digital.intelligence.behaviour import BehaviourService
from telco_digital.intelligence.churn import ChurnService
from telco_digital.intelligence.event_memory import EventMemoryService
from telco_digital.intelligence.event_memory.uow import UnitOfWorkEventMemoryQueries
from telco_digital.intelligence.features import CustomerFeatures, GraphFeatures
from telco_digital.intelligence.features.service import FeatureGroup
from telco_digital.intelligence.recommendations import PlanRepositoryCatalogue, RecommendationService

ROOT = Path(".")
for folder in ("outputs/tables", "outputs/plots", "artifacts"):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
AS_OF = datetime.fromisoformat("2026-08-20T12:00:00+00:00")
CHURN_AS_OF = datetime.fromisoformat("2026-08-21T00:00:00+00:00")


In [2]:
async def features_from_uow(uow, customer_ref, as_of):
    customer = await uow.customers.get_by_ref(customer_ref)
    start_30 = as_of - timedelta(days=30)
    start_90 = as_of - timedelta(days=90)
    start_365 = as_of - timedelta(days=365)
    usage = [row for row in await uow.usage_events.list_as_of(customer.id, as_of) if row.occurred_at >= start_90]
    usage_30 = [row for row in usage if row.occurred_at >= start_30]
    recharges = [row for row in await uow.recharges.list_as_of(customer.id, as_of) if row.occurred_at >= start_90]
    recharge_30 = [row for row in recharges if row.occurred_at >= start_30]
    small = sum(row.amount <= Decimal("500") for row in recharge_30)
    service = [row for row in await uow.service_interactions.list_by_customer(customer.id) if start_90 <= row.occurred_at <= as_of]
    travels = [row for row in await uow.travels.list_as_of(customer.id, as_of) if row.started_at >= start_365]
    return CustomerFeatures(
        customer_id=customer.id,
        customer_ref=customer_ref,
        as_of=as_of,
        computed_at=as_of,
        temporal={
            "usage": FeatureGroup(window_days=30, values={
                "event_count_30d": len(usage_30),
                "data_mb_30d": float(sum((row.data_mb for row in usage_30), Decimal("0"))),
                "data_mb_90d": float(sum((row.data_mb for row in usage), Decimal("0"))),
                "data_mb_change_ratio": None,
            }),
            "recharge": FeatureGroup(window_days=30, values={
                "count_30d": len(recharge_30),
                "amount_30d": float(sum((row.amount for row in recharge_30), Decimal("0"))),
                "small_recharge_count_30d": small,
                "frequent_small_recharge_evidence": small >= 3,
            }),
            "service": FeatureGroup(window_days=90, values={
                "interaction_count_90d": len(service),
                "complaint_count_90d": sum(row.interaction_type == "COMPLAINT" for row in service),
                "open_count": sum(row.status == "OPEN" for row in service),
            }),
            "travel": FeatureGroup(window_days=365, values={"trip_count_365d": len(travels), "roaming_days_365d": 0}),
        },
        graph=GraphFeatures(available=False, values={}),
        provenance=("in-memory seed",),
    )


class UowFeatures:
    def __init__(self, uow):
        self.uow = uow

    async def calculate(self, customer_ref, as_of):
        return await features_from_uow(self.uow, customer_ref, as_of)


uow = InMemoryUnitOfWork()
await seed_demo_customers(uow, clock=FixedClock(AS_OF))
memory = EventMemoryService(UnitOfWorkEventMemoryQueries(uow))
features = UowFeatures(uow)
engine = DecisionEngine(
    RecommendationService(memory, PlanRepositoryCatalogue(uow.plans)),
    BehaviourService(features, memory),
    ChurnService(features),
)


In [3]:
rows = []
cases = [
    ("U001", AS_OF, "SG"),
    ("U002", AS_OF, None),
    ("U004", CHURN_AS_OF, None),
]
for ref, as_of, destination in cases:
    document = await engine.evaluate(ref, as_of, destination=destination)
    rows.append({
        "customer_ref": ref,
        "action": document.action,
        "target_plan_code": document.target_plan_code,
        "reason_codes": ", ".join(document.reason_codes),
        "what": document.explanation.what,
        "why": document.explanation.why,
        "alternatives": ", ".join(document.explanation.alternatives),
        "churn_risk_band": document.churn_risk_band,
    })
table = pd.DataFrame(rows)
table.to_json(ROOT / "outputs" / "tables" / "persona_decisions.json", orient="records", indent=2)
u004 = table[table.customer_ref == "U004"].iloc[0]
no_discount = pd.DataFrame([{
    "customer_ref": "U004",
    "action": u004.action,
    "mentions_discount": "discount" in f"{u004.what} {u004.why}".lower(),
    "invents_fake_plan": "FAKE_PLAN" in f"{u004.what} {u004.why}",
    "percent_20": "20%" in f"{u004.what} {u004.why}",
}])
no_discount.to_json(ROOT / "outputs" / "tables" / "u004_no_discount.json", orient="records", indent=2)
metrics = {
    "u001_action": table.loc[table.customer_ref == "U001", "action"].item(),
    "u001_target": table.loc[table.customer_ref == "U001", "target_plan_code"].item(),
    "u004_action": table.loc[table.customer_ref == "U004", "action"].item(),
    "u002_action": table.loc[table.customer_ref == "U002", "action"].item(),
}
(ROOT / "outputs" / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
table


,customer_ref,action,target_plan_code,reason_codes,what,why,alternatives,churn_risk_band
0,U001,PRESENT_OFFER,ROAM_15,"HISTORICAL_EPISODE, CATALOGUE_MATCH, DURATION_...",Present catalogue offer ROAM_15.,Retrieved travel memory ranks ROAM_15 for the ...,"ROAM_30, ROAM_5",MEDIUM
1,U002,NO_INVENTED_OFFER,None,"PRICE_SENSITIVE, NO_CATALOGUE_TRAVEL_CONTEXT",Do not invent a discount or roam plan.,"PRICE_SENSITIVE evidence is present, but there...",,LOW
2,U004,SUPPORT_FOLLOW_UP,None,"CHURN_HIGH, NETWORK_OR_COMPLAINT, NO_AUTO_DISC...",Follow up on the open service issue. Do not is...,Churn is HIGH with declining engagement or ope...,,HIGH


In [4]:
fig, ax = plt.subplots(figsize=(6, 3))
counts = table["action"].value_counts()
ax.bar(counts.index.astype(str), counts.values, color="#1677ff")
ax.set_title("Decision actions on seed personas")
ax.set_ylabel("Personas")
fig.tight_layout()
fig.savefig(ROOT / "outputs" / "plots" / "decision_actions.png", dpi=120)
plt.close(fig)
counts


action
PRESENT_OFFER        1
NO_INVENTED_OFFER    1
SUPPORT_FOLLOW_UP    1
Name: count, dtype: int64